In [1]:
%pip install lightgbm xgboost catboost


Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import pandas as pd
import joblib


from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.ensemble import ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.neural_network import MLPClassifier

from src.simulator_betting import *
from src.config import *
from src.utils import get_latest_file
pd.set_option("display.max_columns", None)

# --- 1. Charger le modèle



In [3]:

model_path = get_latest_file(DATA_MODELS_DIR)
model = joblib.load(model_path)
print("Modèle chargé:", model_path)


Modèle chargé: data/models/stacking_model_2025-05-26.joblib


# --- 2. Charger les données NBA


In [4]:

dataset_path = get_latest_file(DATA_FINAL_CLEANED_DATASET_DIR)
nba_df = pd.read_csv(dataset_path, dtype={"GAME_ID": str, "GAME_DATE": str, "SEASON": str})

nba_df = nba_df[nba_df['SEASON'] == '2023-24']


print("Dataset NBA chargé:", dataset_path)

nba_df

Dataset NBA chargé: data/final_cleaned_dataset/nba_features_cleaned_final_2025-05-26_17-54-31.csv


,GAME_ID,TEAM_ID,GAME_DATE,OPP_TEAM_ID,SEASON,IS_HOME,IS_WIN,ROLL_HOME_WINRATE_3,ROLL_AWAY_WINRATE_3,ROLL_HOME_WINRATE_5,ROLL_AWAY_WINRATE_5,ROLL_HOME_WINRATE_10,ROLL_AWAY_WINRATE_10,ROLL_HOME_WINRATE_25,ROLL_AWAY_WINRATE_25,ROLL_HOME_WINRATE_50,ROLL_AWAY_WINRATE_50,ROLL_HOME_WINRATE_100,ROLL_AWAY_WINRATE_100,ROLL_HOME_WINRATE_200,ROLL_AWAY_WINRATE_200,ROLL_WIN_RATIO_3,ROLL_WIN_RATIO_5,ROLL_WIN_RATIO_10,ROLL_WIN_RATIO_25,ROLL_WIN_RATIO_50,ROLL_WIN_RATIO_100,ROLL_WIN_RATIO_200,WIN_STREAK,HOME_WIN_STREAK,AWAY_WIN_STREAK,DAYS_SINCE_LAST_GAME,OPP_DAYS_SINCE_LAST_GAME,REST_ADVANTAGE,ROLL_HOME_REST_ADV_3,ROLL_AWAY_REST_ADV_3,ROLL_HOME_REST_ADV_5,ROLL_AWAY_REST_ADV_5,ROLL_HOME_REST_ADV_10,ROLL_AWAY_REST_ADV_10,ROLL_HOME_REST_ADV_25,ROLL_AWAY_REST_ADV_25,ROLL_HOME_REST_ADV_50,ROLL_AWAY_REST_ADV_50,ROLL_HOME_REST_ADV_100,ROLL_AWAY_REST_ADV_100,ROLL_HOME_REST_ADV_200,ROLL_AWAY_REST_ADV_200,H2H_LAST_3_DIFF,H2H_LAST_3_WINRATE,H2H_LAST_3_COUNT,H2H_LAST_5_DIFF,H2H_LAST_5_WINRATE,H2H_LAST_5_COUNT,H2H_LAST_10_DIFF,H2H_LAST_10_WINRATE,H2H_LAST_10_COUNT,H2H_LAST_25_DIFF,H2H_LAST_25_WINRATE,H2H_LAST_25_COUNT,H2H_LAST_50_DIFF,H2H_LAST_50_WINRATE,H2H_LAST_50_COUNT,H2H_LAST_100_DIFF,H2H_LAST_100_WINRATE,H2H_LAST_100_COUNT,H2H_LAST_200_DIFF,H2H_LAST_200_WINRATE,H2H_LAST_200_COUNT,H2H_SEASON_WINS,H2H_SEASON_MATCHES,H2H_SEASON_WINRATE,H2H_WIN_STREAK,ELO_PRE,OPP_ELO_PRE,ELO_PRE_SEASON,OPP_ELO_PRE_SEASON
58976,0022300061,1610612743,2023-10-24,1610612747,2023-24,1,1,1.0,1.000000,0.666667,1.000000,0.8,1.000000,0.923077,0.500000,0.833333,0.500000,0.846154,0.500000,0.700000,0.520000,1.000000,0.8,0.9,0.72,0.66,0.68,0.610,3,6,1,134.0,155.0,-21.0,0.0,0.0,-32.666667,0.0,-44.0,0.0,-35.230769,-12.833333,-69.500000,-39.269231,-109.211538,-96.875000,-105.390000,-91.850000,3,1.000000,3,5,1.0,5,4,0.7,10,1,0.52,25,10,0.60,50,-4,0.480000,100,-8,0.463636,110,0,0,0.000000,5,1688.099683,1571.552579,1500.000000,1500.000000
58977,0022300061,1610612747,2023-10-24,1610612743,2023-24,0,0,0.0,0.000000,0.333333,0.000000,0.6,0.200000,0.750000,0.461538,0.680000,0.480000,0.600000,0.460000,0.564356,0.383838,0.000000,0.2,0.4,0.60,0.58,0.53,0.475,0,0,0,155.0,155.0,0.0,0.0,0.0,0.000000,-61.5,0.0,-35.4,-40.500000,-36.153846,-53.640000,-56.800000,-113.280000,-93.200000,-109.237624,-89.313131,-3,0.000000,3,-5,0.0,5,-4,0.3,10,-1,0.48,25,-10,0.40,50,4,0.520000,100,8,0.536364,110,0,0,0.000000,0,1571.552579,1688.099683,1500.000000,1500.000000
58978,0022300062,1610612744,2023-10-24,1610612756,2023-24,1,0,1.0,0.000000,1.000000,0.000000,0.6,0.400000,0.727273,0.428571,0.800000,0.320000,0.795918,0.294118,0.782178,0.404040,0.333333,0.4,0.5,0.56,0.56,0.54,0.595,0,0,1,165.0,225.0,-60.0,0.0,0.0,0.000000,0.0,-11.2,0.0,-27.363636,-27.500000,-39.600000,-47.400000,-98.795918,-99.000000,-98.118812,-93.666667,-1,0.333333,3,-3,0.2,5,-2,0.4,10,5,0.60,25,14,0.64,50,2,0.511111,90,2,0.511111,90,0,0,0.000000,1,1558.179623,1580.917377,1500.000000,1500.000000
58979,0022300062,1610612756,2023-10-24,1610612744,2023-24,0,1,0.5,0.000000,0.666667,0.000000,0.8,0.400000,0.714286,0.363636,0.692308,0.500000,0.686275,0.387755,0.722772,0.555556,0.333333,0.4,0.6,0.56,0.60,0.54,0.640,0,0,0,166.0,225.0,-59.0,0.0,0.0,0.000000,0.0,0.0,-3.8,-38.785714,-24.909091,-98.807692,-47.875000,-110.274510,-87.591837,-104.643564,-84.313131,1,0.666667,3,3,0.8,5,2,0.6,10,-5,0.40,25,-14,0.36,50,-2,0.488889,90,-2,0.488889,90,0,0,0.000000,0,1580.917377,1558.179623,1500.000000,1500.000000
58980,0022300063,1610612737,2023-10-25,1610612766,2023-24,0,0,0.0,1.000000,0.333333,0.500000,0.4,0.400000,0.538462,0.416667,0.560000,0.480000,0.583333,0.403846,0.620000,0.420000,0.333333,0.4,0.4,0.48,0.52,0.49,0.520,0,2,0,181.0,254.0,-73.0,0.0,0.0,0.000000,0.0,-30.6,-12.6,-85.538462,-17.500000,-88.040000,-79.640000,-118.833333,-89.115385,-100.510000,-94.680000,-1,0.333333,3,-1,0.4,5,0,0.5,10,-7,0.36,25,4,0.54,50,-4,0.475610,82,-4,0.475610,82,0,0,0.000000,0,1518.053186,1413.275295,1500.000000,1500.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,.


# --- 3. Charger la table des équipes

In [5]:

teams_path = get_latest_file(DATA_TEAMS_DIR)
teams_df = pd.read_csv(teams_path)
team_id_map = dict(zip(teams_df["id"], teams_df["full_name"]))


In [6]:
team_id_map

{1610612737: 'Atlanta Hawks',
 1610612738: 'Boston Celtics',
 1610612739: 'Cleveland Cavaliers',
 1610612740: 'New Orleans Pelicans',
 1610612741: 'Chicago Bulls',
 1610612742: 'Dallas Mavericks',
 1610612743: 'Denver Nuggets',
 1610612744: 'Golden State Warriors',
 1610612745: 'Houston Rockets',
 1610612746: 'Los Angeles Clippers',
 1610612747: 'Los Angeles Lakers',
 1610612748: 'Miami Heat',
 1610612749: 'Milwaukee Bucks',
 1610612750: 'Minnesota Timberwolves',
 1610612751: 'Brooklyn Nets',
 1610612752: 'New York Knicks',
 1610612753: 'Orlando Magic',
 1610612754: 'Indiana Pacers',
 1610612755: 'Philadelphia 76ers',
 1610612756: 'Phoenix Suns',
 1610612757: 'Portland Trail Blazers',
 1610612758: 'Sacramento Kings',
 1610612759: 'San Antonio Spurs',
 1610612760: 'Oklahoma City Thunder',
 1610612761: 'Toronto Raptors',
 1610612762: 'Utah Jazz',
 1610612763: 'Memphis Grizzlies',
 1610612764: 'Washington Wizards',
 1610612765: 'Detroit Pistons',
 1610612766: 'Charlotte Hornets'}

# --- 4. Simuler une saison (exemple 2023-2024)


In [7]:

odds_path = os.path.join(DATA_ODDS_HISTORY_DIR, "nba_2023_2024.csv")
odds_df = pd.read_csv(odds_path)

# Fusion des données
merged = match_odds_with_dataset(odds_df, nba_df, team_id_map)
cleaned_merged = clean_merged_matches(merged)

cleaned_merged



Exemples de clés de match dans odds_df (tolérance date):
0    (2024-06-17, boston celtics, dallas mavericks)
1    (2024-06-14, dallas mavericks, boston celtics)
2    (2024-06-12, dallas mavericks, boston celtics)
3    (2024-06-09, boston celtics, dallas mavericks)
4    (2024-06-06, boston celtics, dallas mavericks)
Name: match_key, dtype: object

Exemples de clés de match dans nba_df:
58976     (2023-10-24, denver nuggets, los angeles lakers)
58978    (2023-10-24, golden state warriors, phoenix suns)
58980       (2023-10-25, charlotte hornets, atlanta hawks)
58982     (2023-10-25, indiana pacers, washington wizards)
58984        (2023-10-25, new york knicks, boston celtics)
Name: match_key, dtype: object

Nombre de lignes fusionnées: 5182
Nombre de correspondances réussies: 2624
Nombre de correspondances échouées: 2558

IDs manquants TEAM_ID: []
IDs manquants OPP_TEAM_ID: []


,date,match_key,GAME_ID,TEAM_ID,GAME_DATE,OPP_TEAM_ID,SEASON,IS_HOME,IS_WIN,ROLL_HOME_WINRATE_3,ROLL_AWAY_WINRATE_3,ROLL_HOME_WINRATE_5,ROLL_AWAY_WINRATE_5,ROLL_HOME_WINRATE_10,ROLL_AWAY_WINRATE_10,ROLL_HOME_WINRATE_25,ROLL_AWAY_WINRATE_25,ROLL_HOME_WINRATE_50,ROLL_AWAY_WINRATE_50,ROLL_HOME_WINRATE_100,ROLL_AWAY_WINRATE_100,ROLL_HOME_WINRATE_200,ROLL_AWAY_WINRATE_200,ROLL_WIN_RATIO_3,ROLL_WIN_RATIO_5,ROLL_WIN_RATIO_10,ROLL_WIN_RATIO_25,ROLL_WIN_RATIO_50,ROLL_WIN_RATIO_100,ROLL_WIN_RATIO_200,WIN_STREAK,HOME_WIN_STREAK,AWAY_WIN_STREAK,DAYS_SINCE_LAST_GAME,OPP_DAYS_SINCE_LAST_GAME,REST_ADVANTAGE,ROLL_HOME_REST_ADV_3,ROLL_AWAY_REST_ADV_3,ROLL_HOME_REST_ADV_5,ROLL_AWAY_REST_ADV_5,ROLL_HOME_REST_ADV_10,ROLL_AWAY_REST_ADV_10,ROLL_HOME_REST_ADV_25,ROLL_AWAY_REST_ADV_25,ROLL_HOME_REST_ADV_50,ROLL_AWAY_REST_ADV_50,ROLL_HOME_REST_ADV_100,ROLL_AWAY_REST_ADV_100,ROLL_HOME_REST_ADV_200,ROLL_AWAY_REST_ADV_200,H2H_LAST_3_DIFF,H2H_LAST_3_WINRATE,H2H_LAST_3_COUNT,H2H_LAST_5_DIFF,H2H_LAST_5_WINRATE,H2H_LAST_5_COUNT,H2H_LAST_10_DIFF,H2H_LAST_10_WINRATE,H2H_LAST_10_COUNT,H2H_LAST_25_DIFF,H2H_LAST_25_WINRATE,H2H_LAST_25_COUNT,H2H_LAST_50_DIFF,H2H_LAST_50_WINRATE,H2H_LAST_50_COUNT,H2H_LAST_100_DIFF,H2H_LAST_100_WINRATE,H2H_LAST_100_COUNT,H2H_LAST_200_DIFF,H2H_LAST_200_WINRATE,H2H_LAST_200_COUNT,H2H_SEASON_WINS,H2H_SEASON_MATCHES,H2H_SEASON_WINRATE,H2H_WIN_STREAK,ELO_PRE,OPP_ELO_PRE,ELO_PRE_SEASON,OPP_ELO_PRE_SEASON,TEAM_NAME,OPPONENT_NAME,ODDS,OPP_ODDS
0,2024-06-18,"(2024-06-17, boston celtics, dallas mavericks)",0042300405,1610612738,2024-06-17,1610612742,2023-24,1.0,1.0,1.000000,0.500000,1.000000,0.666667,1.000000,0.800000,0.812500,0.777778,0.880000,0.760000,0.882353,0.693878,0.794118,0.653061,0.666667,0.8,0.9,0.80,0.82,0.79,0.725,0.0,0.0,5.0,3.0,3.0,0.0,0.0,0.000000,-43.5,0.000000,-38.600000,0.000000,-38.375000,-2.000000,-45.120000,-61.120000,-93.568627,-120.020408,-93.401961,-112.316327,1.0,0.666667,3.0,3.0,0.8,5.0,4.0,0.7,10.0,3.0,0.56,25.0,-8.0,0.420000,50.0,-10.0,0.403846,52.0,-10.0,0.403846,52.0,5.0,6.0,0.833333,0.0,1767.087143,1677.852091,1753.902270,1677.339943,boston celtics,dallas mavericks,1.32,2.93
1,2024-06-18,"(2024-06-17, boston celtics, dallas mavericks)",0042300405,1610612742,2024-06-17,1610612738,2023-24,0.0,0.0,0.500000,0.000000,0.500000,0.333333,0.600000,0.600000,0.545455,0.642857,0.695652,0.629630,0.600000,0.600000,0.602041,0.480392,0.333333,0.4,0.6,0.60,0.66,0.60,0.540,1.0,0.0,1.0,3.0,3.0,0.0,0.0,0.000000,0.0,-30.000000,0.000000,-39.600000,-2.909091,-39.214286,-37.956522,-68.814815,-101.160000,-108.480000,-113.316327,-94.235294,-1.0,0.333333,3.0,-3.0,0.2,5.0,-4.0,0.3,10.0,-3.0,0.44,25.0,8.0,0.580000,50.0,10.0,0.596154,52.0,10.0,0.596154,52.0,1.0,6.0,0.166667,1.0,1677.852091,1767.087143,1677.339943,1753.902270,dallas mavericks,boston celtics,2.93,1.32
2,2024-06-15,"(2024-06-14, dallas mavericks, boston celtics)",0042300404,1610612738,2024-06-14,1610612742,2023-24,0.0,0.0,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.812500,0.888889,0.884615,0.791667,0.865385,0.708333,0.794118,0.663265,1.000000,1.0,1.0,0.84,0.84,0.79,0.730,10.0,11.0,5.0,2.0,2.0,0.0,-43.5,0.000000,-43.5,0.000000,-38.600000,0.000000,-38.375000,-16.555556,-46.230769,-63.666667,-91.769231,-122.520833,-93.401961,-113.765306,3.0,1.000000,3.0,5.0,1.0,5.0,4.0,0.7,10.0,3.0,0.56,25.0,-8.0,0.420000,50.0,-9.0,0.411765,51.0,-9.0,0.411765,51.0,5.0,5.0,1.000000,7.0,1783.097754,1662.340023,1769.493132,1662.245687,boston celtics,dallas mavericks,1.76,1.87
3,2024-06-15,"(2024-06-14, dallas mavericks, boston celtics)",0042300404,1610612742,2024-06-14,1610612738,2023-24,1.0,1.0,0.000000,0.000000,0.000000,0.333333,0.500000,0.666667,0.545455,0.642857,0.695652,0.629630,0.591837,0.607843,0.597938,0.485437,0.000000,0.2,0.6,0.60,0.66,0.60,0.540,0.0,0.0,0.0,2.0,2.0,0.0,0.0,-45.000000,0.0,-30.000000,0.000000,-33.000000,-3.363636,-39.214286,-40.913043,-68.814815,-103.224490,-110.686275,-114.484536,-93.320388,-3.0,0.000000,3.0,-5.0,0.0,5.0,-4.0,0.3,10.0,-3.0,0.44,25.0,8.0,0.580000

In [8]:
# Simulation
bets = simulate_bets(
    merged_df=merged,
    model_pipeline=model,
    min_ev=0.05,
    bankroll=1000,
    max_risk=0.05
)



KeyError: "['ROLL_HOME_REST_ADV_3'] not in index"

In [ ]:

# Évaluation
results = evaluate_simulation(bets)
print("\nRésultats de la simulation 2023-2024:")
for k, v in results.items():
    print(f"{k}: {v}")

# Aperçu des paris
bets.head()